# Generate WarpX input files

Two kinds of files:

- **A. Benchmark variants:** head-on round-beam inputs from `make_inputs.headon_round_inputs`, scanned over
  any of its parameters. The manifest includes the analytic prediction and whether every validity check passes,
  both read back from each written file.
- **B. Variants of an existing input file:** e.g. the FCC-ee box-scan input, with keys replaced, added or
  removed (`make_inputs.from_template`).

Each scan writes one file per grid point and a `manifest.csv` into `generated_inputs/` (git-ignored), and prints
the `sbatch` commands.

In [ ]:
import csv, os, sys
import numpy as np

REPO = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(REPO, "benchmarks")); sys.path.insert(0, REPO)
import make_inputs as mi
import bhabha_theory as bt
import wx_bhabha as wx
PERLMUTTER_REPO = "$HOME/src/bhlumi-warpx"

def on_nersc(path):
    rel = os.path.relpath(path, REPO)
    return f"{PERLMUTTER_REPO}/{rel}" if not rel.startswith("..") else path

def show(rows):
    bt.print_table([[str(r.get(k, "")) for k in rows[0]] for r in rows], list(rows[0]))
print(mi.HEADON_DEFAULTS)

## A. Benchmark variants

`GRID` keys are `HEADON_DEFAULTS` names; `FIXED` sets other parameters for every file. The summary columns come
from reading each file back, so they are exactly what will be run. A file with failed checks still runs, but
the comparison would then test something other than the formula (e.g. disruption or too few slices).

In [ ]:
GRID = {"npart": [1e10, 2e10, 4e10], "sigma_um": [5.0, 10.0]}
FIXED = {"sigmaz_mm": 1.0, "nz": 1024}
OUT_A = os.path.join(REPO, "generated_inputs", "headon")
THETA_MIN, THETA_MAX, THETA_REF = 0.040, 0.120, 0.070

def summary_headon(path):
    p = bt.Params(path)
    failed = [c[0] for c in bt.validity_checks(p, THETA_MIN, THETA_MAX, mc_events=1000) if not c[3]]
    return {"D": f"{bt.disruption(p):.3g}", "L_BX [m^-2]": f"{bt.luminosity_per_crossing(p):.3g}",
            f"kick@{THETA_REF*1e3:.0f}mrad [urad]": f"{bt.kick_closed_form(THETA_REF, p)*1e6:.3f}",
            "failed checks": "; ".join(failed) or "none"}

rows_a = mi.scan(GRID, OUT_A, fixed=FIXED, name="inputs", summary=summary_headon)
show(rows_a)
print()
for r in rows_a:
    tag = "headon_" + os.path.splitext(r["file"])[0].replace("inputs_", "")
    print(f"sbatch --export=ALL,INPUTS={on_nersc(os.path.join(OUT_A, r['file']))},TAG={tag} benchmarks/submit_benchmark.sh")

## B. Variants of an existing input file

`BASE` is any WarpX input file; `GRID` and `FIXED` keys are WarpX keys. `REMOVE` drops every key starting with
one of the prefixes; the default removes the external solenoid field from the box-scan input. The summary uses
`wx_bhabha.py`'s bunch model for the luminosity per crossing (hourglass and crossing angle included).

In [ ]:
BASE = os.path.join(REPO, "examples", "fcc_z_boxscan_template.txt")
GRID = {"my_constants.crossing_angle": ["0.", "15e-3", "30e-3"]}
FIXED = {}
REMOVE = ("particles.B_ext_particle_init_style", "particles.Bx_external_particle_function",
          "particles.By_external_particle_function", "particles.Bz_external_particle_function")
OUT_B = os.path.join(REPO, "generated_inputs", "fcc_variants")

def summary_template(path):
    c = wx.eval_constants(wx.read_inputs(path))
    b = wx.Bunches(c)
    has_B = any("external_particle_function" in k for k, _ in wx.read_inputs(path))
    return {"half crossing [rad]": f"{b.phi:.4g}", "sigma_x [um]": f"{b.sx*1e6:.4g}", "sigma_y [nm]": f"{b.sy*1e9:.4g}",
            "L_BX geometric [m^-2]": f"{b.geometric_lumi():.4g}", "external B": has_B}

if os.path.isfile(BASE):
    rows_b = mi.scan(GRID, OUT_B, base=BASE, fixed=FIXED, remove_prefixes=REMOVE, name="inputs", summary=summary_template)
    show(rows_b)
    print()
    for r in rows_b:
        tag = os.path.splitext(r["file"])[0].replace("inputs_", "")
        print(f"sbatch --export=ALL,INPUTS={on_nersc(os.path.join(OUT_B, r['file']))},TAG={tag},BZ=0 submit_wx_bhabha.sh")
else:
    print(f"{BASE} not found")

Notes:
- `generated_inputs/` is git-ignored. On Perlmutter, run this notebook inside the clone so the printed paths are
  right, or copy the files there.
- For section B, `submit_wx_bhabha.sh` still passes `--set my_constants.Bz=$BZ`. With the external-field keys
  removed that constant is unused, so it has no effect.